# TDA Feature Extraction (Notebook 1)
This notebook discovers sessions, computes Vietoris–Rips persistence diagrams, and extracts a **compact, interpretable set of topological features** for downstream modeling.

**Outputs**:
- `features_{label}_{source}.parquet` (and CSV) saved to `OUTPUT_DIR`
- Log printouts with class counts

**Tip:** Adjust the config cell below for your environment.

In [1]:

# ================== 0) Setup & Config ==================
import os, gc, json, random
from glob import glob
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# plotting not required in this notebook; keep imports minimal
from sklearn.decomposition import PCA
from scipy import stats

# giotto-tda
from gtda.homology import VietorisRipsPersistence

# ---------- Your env (EDIT ME) ----------
DATA_DIR   = "/proj/leelab/projects/Scott/Embeddings_All/output_embeddings"
OUTPUT_DIR = "/proj/leelab/projects/Scott/Embeddings_All/tda_feature_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Labels (session-wise target definition is below)
LABEL_DICT = {
    "sleep_label": 5,   # multiclass
    "desat_label": 2,
    "eeg_label": 2,
    "apnea_label": 2,
    "hypop_label": 2,
}

# Runtime controls
LABEL_NAME       = "sleep_label"  # <-- choose label
SOURCE           = "emb"        # "point" (PHATE) or "emb"
EMB_PCA_DIM      = 10             # only used if SOURCE == "emb"
MAX_SESSIONS     = -1             # -1 = use ALL valid sessions; otherwise a max cap (balanced per class)
MAX_T_PER_SESS   = 1500           # cap timepoints per session
SUBSAMPLE_STRIDE = 2              # take every k-th point
HOMOLOGY_DIMS    = [0, 1]
RIPS_MAX_EDGE    = np.inf         # let ripser choose by default (np.inf allowed)
SEED             = 42
random.seed(SEED); np.random.seed(SEED)

print(f"[config] DATA_DIR={DATA_DIR}\nOUTPUT_DIR={OUTPUT_DIR}\nLABEL_NAME={LABEL_NAME}\nSOURCE={SOURCE}")


[config] DATA_DIR=/proj/leelab/projects/Scott/Embeddings_All/output_embeddings
OUTPUT_DIR=/proj/leelab/projects/Scott/Embeddings_All/tda_feature_outputs
LABEL_NAME=sleep_label
SOURCE=emb


In [2]:

# ================== 1) IO helpers ==================
def discover_pairs(data_dir: str) -> List[str]:
    embs = sorted(glob(os.path.join(data_dir, "*_embeddings.npy")))
    # IDs look like "<i>_<j>_embeddings.npy"
    pair_ids = [os.path.basename(f).replace("_embeddings.npy", "") for f in embs]
    pair_ids = sorted(pair_ids, key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1])))
    return pair_ids

def session_paths(pid: str, label_name: str) -> Dict[str, str]:
    return {
        "emb":   os.path.join(DATA_DIR, f"{pid}_embeddings.npy"),
        "point": os.path.join(DATA_DIR, f"{pid}_phate_point_feature_normalized.npy"),
        "y":     os.path.join(DATA_DIR, f"{pid}_{label_name}.npy"),
    }

def valid_session(pid: str, label_name: str) -> bool:
    p = session_paths(pid, label_name)
    return all(os.path.exists(pth) for pth in p.values())

def load_session_matrix(pid: str, source: str, emb_pca_dim: int) -> np.ndarray:
    """Returns X (T x D_source). If SOURCE == 'emb', applies PCA->emb_pca_dim for tractable VR."""
    p = session_paths(pid, LABEL_NAME)
    if source == "point":
        X = np.load(p["point"])  # shape (T, d_phate)
    else:
        X = np.load(p["emb"])    # (T, d_emb)
        if emb_pca_dim is not None and X.shape[1] > emb_pca_dim:
            X = PCA(n_components=emb_pca_dim, random_state=SEED).fit_transform(X)
    return np.asarray(X, dtype=np.float32)

def label_for_session(pid: str, label_name: str, num_classes: int) -> int:
    """Session-wise label by majority over time."""
    p = session_paths(pid, label_name)
    y = np.load(p["y"])
    if num_classes == 2:
        y_bin = (y > 0).astype(int)
        return int(np.round(y_bin.mean()))
    else:
        counts = np.bincount(y.astype(int), minlength=num_classes)
        return int(np.argmax(counts))


In [3]:

# ================== 2) Subsampling for stability & memory ==================
def subsample_trajectory(X: np.ndarray,
                         max_t: int = MAX_T_PER_SESS,
                         stride: int = SUBSAMPLE_STRIDE) -> np.ndarray:
    if stride > 1:
        X = X[::stride]
    if X.shape[0] > max_t:
        idx = np.linspace(0, X.shape[0] - 1, max_t).astype(int)
        X = X[idx]
    return X


In [4]:

# ================== 3) TDA primitives ==================
VR = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=HOMOLOGY_DIMS,
    collapse_edges=True,
    max_edge_length=RIPS_MAX_EDGE,
    n_jobs=1,
)

def compute_diagram(X: np.ndarray) -> np.ndarray:
    """Returns a diagram array D of shape (n_bars, 3): [birth, death, hom_dim]"""
    return VR.fit_transform([X])[0]


In [5]:

# ================== 4) Feature engineering ==================
def lifetime_stats(D: np.ndarray, h: int, eps=1e-8):
    sel = D[D[:,2]==h]
    if sel.size == 0:
        return dict(n_bars=0.0, sum_pers=0.0, max_pers=0.0,
                    mean_midlife=0.0, max_birth=0.0, max_death=0.0, entropy_pers=0.0)
    life  = (sel[:,1] - sel[:,0]).clip(min=0.0)
    birth = sel[:,0]; death = sel[:,1]
    p = life / (life.sum() + eps)
    return dict(
        n_bars=float(len(life)),
        sum_pers=float(life.sum()),
        max_pers=float(life.max()) if len(life) else 0.0,
        mean_midlife=float(np.mean(0.5*(birth+death))),
        max_birth=float(birth.max() if len(birth) else 0.0),
        max_death=float(death.max() if len(death) else 0.0),
        entropy_pers=float(stats.entropy(p + eps)),
    )

def betti_curve_dim(D: np.ndarray, dim=1, n_bins=100, t_min=None, t_max=None):
    sel = D[D[:,2]==dim]
    if sel.size == 0:
        if t_min is None or t_max is None:
            return np.array([]), np.array([])
        t = np.linspace(t_min, t_max, n_bins)
        return t, np.zeros_like(t)
    births = sel[:,0]; deaths = sel[:,1]
    if t_min is None: t_min = float(np.min(births))
    if t_max is None: t_max = float(np.max(deaths))
    if not np.isfinite(t_min) or not np.isfinite(t_max) or t_max <= t_min:
        t = np.linspace(0,1,n_bins); return t, np.zeros_like(t)
    t = np.linspace(t_min, t_max, n_bins)
    counts = np.array([np.sum((births <= ti) & (deaths > ti)) for ti in t], dtype=float)
    return t, counts

def pi_h1_band_energies_hist(D: np.ndarray, bins=(40,40)):
    """Histogram 'PI-like' energies on (birth, persistence) for H1 with weights=persistence."""
    sel = D[D[:,2]==1]
    if sel.size == 0:
        return dict(PI_H1_short=0.0, PI_H1_long=0.0, PI_H1_long_short_ratio=np.nan)
    b = sel[:,0]; p = (sel[:,1]-sel[:,0]).clip(min=0.0)
    bmin,bmax = float(np.min(b)), float(np.max(b))
    pmin,pmax = 0.0, float(np.max(p)) if np.isfinite(np.max(p)) else 1.0
    if not np.isfinite(bmin) or not np.isfinite(bmax) or bmax<=bmin or not np.isfinite(pmax) or pmax<=0:
        return dict(PI_H1_short=np.nan, PI_H1_long=np.nan, PI_H1_long_short_ratio=np.nan)
    H, xedges, yedges = np.histogram2d(b, p, bins=bins, range=[[bmin,bmax],[pmin,pmax]], weights=p)
    H = H.T
    Hh, Wh = H.shape
    band_px = max(2, int(min(Hh,Wh)*0.08))
    short_mask = np.zeros_like(H, dtype=bool)
    for i in range(min(Hh,Wh)):
        j = i
        i0 = max(0, i-band_px); i1=min(Hh, i+band_px+1)
        j0 = max(0, j-band_px); j1=min(Wh, j+band_px+1)
        short_mask[i0:i1, j0:j1] = True
    long_mask = np.zeros_like(H, dtype=bool)
    long_mask[:band_px, :] = True
    long_mask &= ~short_mask
    e_short = float(H[short_mask].sum())
    e_long  = float(H[long_mask].sum())
    ratio   = e_long / (e_short + 1e-8)
    return dict(PI_H1_short=e_short, PI_H1_long=e_long, PI_H1_long_short_ratio=ratio)

def extract_features_from_diagram(D: np.ndarray):
    out = {}
    H0 = lifetime_stats(D, 0); H1 = lifetime_stats(D, 1)
    for k,v in H0.items(): out[f"H0_{k}"] = v
    for k,v in H1.items(): out[f"H1_{k}"] = v
    eps=1e-8
    out["ratio_sum_H1_H0"]   = out["H1_sum_pers"]/(out["H0_sum_pers"]+eps)
    out["ratio_count_H1_H0"] = out["H1_n_bars"]/(out["H0_n_bars"]+eps)
    out["midlife_gap_H1_H0"] = out["H1_mean_midlife"] - out["H0_mean_midlife"]
    out["entropy_gap_H1_H0"] = out["H1_entropy_pers"] - out["H0_entropy_pers"]
    # Betti H1 curve scalars
    t, bc1 = betti_curve_dim(D, dim=1, n_bins=100)
    if bc1.size:
        out["BettiH1_AUC"] = float(np.trapz(bc1, t))
        out["BettiH1_peak_loc_norm"] = float((t[np.argmax(bc1)] - t.min())/(t.ptp()+1e-8))
        out["betti_L1"] = float(np.linalg.norm(bc1, 1))
        out["betti_L2"] = float(np.linalg.norm(bc1, 2))
    else:
        out["BettiH1_AUC"] = 0.0; out["BettiH1_peak_loc_norm"] = np.nan
        out["betti_L1"] = 0.0; out["betti_L2"] = 0.0
    # H1 histogram PI energies
    out.update(pi_h1_band_energies_hist(D))
    return out


In [6]:

# ================== 5) Build session table (ALL valid sessions or capped) ==================
def build_session_table(label_name: str, num_classes: int, source: str,
                        max_sessions: int = MAX_SESSIONS) -> pd.DataFrame:
    pids = discover_pairs(DATA_DIR)
    rows = []
    for pid in pids:
        if not valid_session(pid, label_name):
            continue
        y_sess = label_for_session(pid, label_name, num_classes)
        rows.append({"pid": pid, "y_sess": y_sess})
    if len(rows) == 0:
        return pd.DataFrame(columns=["pid", "y_sess"])
    df = pd.DataFrame(rows)
    if max_sessions is None or max_sessions < 0:
        return df.reset_index(drop=True)
    # balanced cap per class
    out_idx = []
    for c in sorted(df["y_sess"].unique()):
        idx = df.index[df["y_sess"]==c].tolist()
        random.shuffle(idx)
        take = max_sessions // max(1, len(df["y_sess"].unique()))
        out_idx.extend(idx[:take])
    return df.loc[out_idx].reset_index(drop=True)

num_classes = LABEL_DICT[LABEL_NAME]
tab = build_session_table(LABEL_NAME, num_classes, SOURCE, MAX_SESSIONS)
print(f"[info] Using {len(tab)} sessions; class counts:\n" + tab['y_sess'].value_counts().sort_index().to_string())


[info] Using 2522 sessions; class counts:
y_sess
0     346
2    1932
3     164
4      80


In [ ]:

# ================== 6) Extract features over sessions ==================
feats = []
for i, row in tab.iterrows():
    pid, cls = row["pid"], int(row["y_sess"])
    X = load_session_matrix(pid, SOURCE, EMB_PCA_DIM)
    X = subsample_trajectory(X, MAX_T_PER_SESS, SUBSAMPLE_STRIDE)
    D = compute_diagram(X)
    f = {"pid": pid, "y_sess": cls, "label_name": LABEL_NAME, "source": SOURCE}
    f.update(extract_features_from_diagram(D))
    feats.append(f)
    if (i+1) % 50 == 0:
        print(f"[feat] {i+1}/{len(tab)}")
    del X, D; gc.collect()

df = pd.DataFrame(feats)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
parquet_path = os.path.join(OUTPUT_DIR, f"features_{LABEL_NAME}_{SOURCE}.parquet")
csv_path     = os.path.join(OUTPUT_DIR, f"features_{LABEL_NAME}_{SOURCE}.csv")
df.to_parquet(parquet_path)
df.to_csv(csv_path, index=False)

print(f"[save] Features ->\n  {parquet_path}\n  {csv_path}")
print(df.head())


[feat] 50/2522
[feat] 100/2522
[feat] 150/2522
[feat] 200/2522
[feat] 250/2522
[feat] 300/2522
[feat] 350/2522
[feat] 400/2522
[feat] 450/2522
[feat] 500/2522
[feat] 550/2522
[feat] 600/2522
[feat] 650/2522
